# FedAvg From Scratch — IID MNIST Baseline

This notebook implements **Federated Averaging (FedAvg)** in PyTorch without a federated-learning framework, then studies how the number of local epochs \(E\) changes convergence under an IID client partition.

**Research question.** With communication rounds held fixed, how does increasing local computation \(E\) affect:
- global test accuracy and loss,
- communication rounds needed to reach target accuracy,
- cumulative local computation?

The checked-in baseline uses 5 clients, full participation, MNIST, SGD, batch size 64, learning rate 0.01, and \(E\in\{1,5,10\}\).

> The full sweep is intentionally **not run automatically** because it took over an hour on Colab CPU. Saved results are loaded near the end of the notebook.

## 1. Setup

In [ ]:
import copy
import json
import random
import time
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchvision import datasets, transforms

In [ ]:
@dataclass
class ExperimentConfig:
    seed: int = 42
    num_clients: int = 5
    num_rounds: int = 20
    batch_size: int = 64
    learning_rate: float = 0.01
    local_epoch_values: tuple = (1, 5, 10)
    datasets_dir: str = "datasets"
    results_dir: str = "results/iid_baseline"


CFG = ExperimentConfig()


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_best_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


device = get_best_device()
print(f"Using device: {device}")

## 2. Dataset, model, and core training utilities

In [ ]:
def load_mnist(root: str):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,)),
    ])
    train_ds = datasets.MNIST(root=root, train=True, download=True, transform=transform)
    test_ds = datasets.MNIST(root=root, train=False, download=True, transform=transform)
    return train_ds, test_ds


class SimpleMLP(nn.Module):
    def __init__(self, num_inputs: int = 28 * 28, num_classes: int = 10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(num_inputs, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.net(x)


def train_one_epoch(model, loader, optimizer, loss_fn, device):
    model.train()
    total_loss = 0.0
    total_examples = 0

    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        optimizer.step()

        batch_size = yb.size(0)
        total_loss += loss.item() * batch_size
        total_examples += batch_size

    return total_loss / total_examples


def evaluate(model, loader, loss_fn, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    with torch.inference_mode():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)

            logits = model(xb)
            loss = loss_fn(logits, yb)

            batch_size = yb.size(0)
            total_loss += loss.item() * batch_size
            total_correct += (logits.argmax(dim=1) == yb).sum().item()
            total_examples += batch_size

    return total_loss / total_examples, total_correct / total_examples

### Optional centralized sanity check

A centralized run is useful only to verify that the model and optimization pipeline can learn MNIST. It is **not used to initialize the federated model**.

The original scratch notebook selected a checkpoint using test-set loss. That is removed here: the test set is evaluation-only.

In [ ]:
def train_centralized(model, train_loader, test_loader, epochs, learning_rate, device):
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
    history = []

    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn, device)
        test_loss, test_acc = evaluate(model, test_loader, loss_fn, device)
        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "test_loss": test_loss,
            "test_accuracy": test_acc,
        })

    return model, pd.DataFrame(history)


RUN_CENTRALIZED_SANITY_CHECK = False

## 3. Create 5 IID clients

In [ ]:
def partition_iid(train_ds, num_clients: int, seed: int):
    generator = torch.Generator().manual_seed(seed)
    shuffled = torch.randperm(len(train_ds), generator=generator)
    splits = torch.tensor_split(shuffled, num_clients)
    return {client_id: split.clone() for client_id, split in enumerate(splits)}


def verify_partition(client_indices, dataset_size: int):
    all_indices = torch.cat(list(client_indices.values()))
    assert all_indices.numel() == dataset_size, "Some examples were dropped or duplicated."
    assert torch.unique(all_indices).numel() == dataset_size, "Client partitions overlap."
    assert all_indices.min().item() >= 0
    assert all_indices.max().item() < dataset_size


def create_client_loaders(train_ds, client_indices, batch_size: int):
    return {
        client_id: torch.utils.data.DataLoader(
            torch.utils.data.Subset(train_ds, indices),
            batch_size=batch_size,
            shuffle=True,
        )
        for client_id, indices in client_indices.items()
    }


def client_label_counts(train_ds, client_indices):
    rows = []
    targets = train_ds.targets

    for client_id, indices in client_indices.items():
        counts = torch.bincount(targets[indices], minlength=10)
        row = {"client": client_id, "samples": len(indices)}
        row.update({f"digit_{d}": int(counts[d]) for d in range(10)})
        rows.append(row)

    return pd.DataFrame(rows)

In [ ]:
set_seed(CFG.seed)
train_ds, test_ds = load_mnist(CFG.datasets_dir)

client_indices = partition_iid(train_ds, CFG.num_clients, CFG.seed)
verify_partition(client_indices, len(train_ds))

client_loaders = create_client_loaders(train_ds, client_indices, CFG.batch_size)
test_loader = torch.utils.data.DataLoader(
    test_ds,
    batch_size=CFG.batch_size,
    shuffle=False,
)

client_label_counts(train_ds, client_indices)

With 60,000 MNIST training examples and 5 clients, each client receives 12,000 examples. The IID split randomly assigns examples, so each client's digit distribution should be approximately similar.

## 4. Local client update and FedAvg aggregation

In [ ]:
def client_update(
    global_model: nn.Module,
    client_loader,
    local_epochs: int,
    learning_rate: float,
    loss_fn: nn.Module,
    device,
):
    local_model = copy.deepcopy(global_model).to(device)
    optimizer = torch.optim.SGD(local_model.parameters(), lr=learning_rate)

    for _ in range(local_epochs):
        train_one_epoch(local_model, client_loader, optimizer, loss_fn, device)

    return local_model


def model_distance(model_a: nn.Module, model_b: nn.Module) -> float:
    with torch.no_grad():
        vec_a = torch.nn.utils.parameters_to_vector(model_a.parameters())
        vec_b = torch.nn.utils.parameters_to_vector(model_b.parameters())
        return torch.dist(vec_a, vec_b, p=2).item()


def aggregate(
    global_model: nn.Module,
    local_models: list[nn.Module],
    client_sizes: list[int],
) -> nn.Module:
    """Weighted FedAvg aggregation for models whose trainable state is in parameters.

    SimpleMLP has no BatchNorm-style buffers, so parameter-wise in-place
    accumulation is sufficient here.
    """
    assert len(local_models) == len(client_sizes) > 0

    total_size = sum(client_sizes)
    global_params = list(global_model.parameters())

    with torch.no_grad():
        for param in global_params:
            param.zero_()

        for local_model, size in zip(local_models, client_sizes):
            weight = size / total_size
            for global_param, local_param in zip(global_params, local_model.parameters()):
                global_param.add_(local_param, alpha=weight)

    return global_model

For client \(k\) with \(n_k\) examples, FedAvg computes

\[
w_{t+1}=\sum_k \frac{n_k}{\sum_j n_j}w_{t+1}^{(k)}.
\]

Because all IID clients here have 12,000 examples, the weights are all \(1/5\). The implementation remains weighted so it still works when client sizes differ later.

## 5. Full federated training loop

In [ ]:
def federated_train(
    global_model,
    client_loaders,
    num_rounds,
    local_epochs,
    learning_rate,
    loss_fn,
    test_loader,
    device,
    verbose=False,
):
    history = []

    test_loss, test_acc = evaluate(global_model, test_loader, loss_fn, device)
    history.append({
        "round": 0,
        "test_loss": test_loss,
        "test_accuracy": test_acc,
    })

    for t in range(num_rounds):
        local_models = []
        client_sizes = []

        # C = 1.0: every client participates in every round.
        for client_loader in client_loaders.values():
            local_model = client_update(
                global_model,
                client_loader,
                local_epochs,
                learning_rate,
                loss_fn,
                device,
            )
            local_models.append(local_model)
            client_sizes.append(len(client_loader.dataset))

        global_model = aggregate(global_model, local_models, client_sizes)

        test_loss, test_acc = evaluate(global_model, test_loader, loss_fn, device)
        history.append({
            "round": t + 1,
            "test_loss": test_loss,
            "test_accuracy": test_acc,
        })

        if verbose:
            print(
                f"Round {t + 1:2d}/{num_rounds} | "
                f"loss={test_loss:.4f} | acc={test_acc:.4f}"
            )

    return global_model, pd.DataFrame(history)

## 6. Controlled experiment: effect of local epochs \(E\)

All \(E\) values must use the **same client partition** and **same initial global weights \(w_0\)**. This isolates \(E\) as the experimental variable.

The full sweep below is expensive on CPU, so it is disabled by default. Set `RUN_FULL_SWEEP = True` only when you intentionally want to reproduce the experiment.

In [ ]:
def run_e_sweep(
    initial_state,
    client_loaders,
    test_loader,
    e_values,
    num_rounds,
    learning_rate,
    device,
):
    histories = {}
    trained_models = {}
    runtimes = {}

    for E in e_values:
        print(f"Running E={E}...")
        set_seed(CFG.seed)

        global_model = SimpleMLP().to(device)
        global_model.load_state_dict(copy.deepcopy(initial_state))

        start = time.perf_counter()
        trained_model, history_df = federated_train(
            global_model=global_model,
            client_loaders=client_loaders,
            num_rounds=num_rounds,
            local_epochs=E,
            learning_rate=learning_rate,
            loss_fn=nn.CrossEntropyLoss(),
            test_loader=test_loader,
            device=device,
            verbose=True,
        )
        runtimes[E] = time.perf_counter() - start
        histories[E] = history_df
        trained_models[E] = trained_model

    return histories, trained_models, runtimes


set_seed(CFG.seed)
base_model = SimpleMLP().to(device)
initial_state = copy.deepcopy(base_model.state_dict())

RUN_FULL_SWEEP = False

if RUN_FULL_SWEEP:
    histories, trained_models, runtimes = run_e_sweep(
        initial_state=initial_state,
        client_loaders=client_loaders,
        test_loader=test_loader,
        e_values=CFG.local_epoch_values,
        num_rounds=CFG.num_rounds,
        learning_rate=CFG.learning_rate,
        device=device,
    )
else:
    print("Full sweep skipped. Saved baseline results are loaded below.")

## 7. Saved IID baseline results

The checked-in run used one seed (`42`). These are baseline results, not yet multi-seed research estimates.

| E | Final test accuracy | Rounds to 90% | Local epochs to 90% | Rounds to 95% | Local epochs to 95% |
|---:|---:|---:|---:|---:|---:|
| 1 | 92.33% | 7 | 7 | — | — |
| 5 | 96.29% | 2 | 10 | 12 | 60 |
| 10 | 97.25% | 1 | 10 | 6 | 60 |

**Interpretation.** Larger \(E\) strongly improves convergence per communication round under IID data, but it also increases local computation. For example, \(E=5\) and \(E=10\) both required about 60 cumulative local epochs per client to reach 95%, while \(E=10\) used only half as many communication rounds.

In [ ]:
RESULT_DIR = Path(CFG.results_dir)

saved_histories = {
    E: pd.read_csv(RESULT_DIR / f"fedavg_iid_E{E}_history.csv")
    for E in CFG.local_epoch_values
}
summary_df = pd.read_csv(RESULT_DIR / "summary.csv")

summary_df

In [ ]:
plt.figure(figsize=(8, 5))
for E, history_df in saved_histories.items():
    plt.plot(
        history_df["round"],
        history_df["test_accuracy"],
        marker="o",
        label=f"E={E}",
    )
plt.xlabel("Communication Round")
plt.ylabel("Test Accuracy")
plt.title("Effect of Local Epochs E on FedAvg (IID MNIST)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
for E, history_df in saved_histories.items():
    plt.plot(
        history_df["round"] * E,
        history_df["test_accuracy"],
        marker="o",
        label=f"E={E}",
    )
plt.xlabel("Cumulative Local Epochs per Client")
plt.ylabel("Test Accuracy")
plt.title("FedAvg Accuracy vs Local Computation (IID MNIST)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Limitations and next experiment

This baseline intentionally answers only a narrow question.

**Current limitations**
- one random seed,
- IID clients only,
- full client participation (\(C=1\)),
- cumulative local epochs are only a proxy for computation; they are not FLOPs or energy,
- the simple MLP has no non-parameter buffers, so the current aggregator is not yet a general-purpose FL framework.

**Next experiment**
1. Replace the IID partition with a controlled non-IID partition.
2. Repeat a small \(E\in\{1,5\}\) development run first.
3. Measure client update magnitude/direction under heterogeneity.
4. Then implement FedProx and compare it with FedAvg under the same initialization and partition.